In [8]:
import shutil
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import os

from shapely.geometry import Point

from hydromt_sfincs import SfincsModel  # noqa: E402

In [9]:
include_gdf  = gpd.read_file("/home/cassandra/Snohomish/2026-04-13/include.geojson")
offshore_gdf = gpd.read_file("/home/cassandra/Snohomish/2026-04-13/offshore_boundary.geojson")
levees_gdf   = gpd.read_file("/home/cassandra/Snohomish/2026-04-13/levees.geojson")
drainage_gdf = gpd.read_file("/home/cassandra/Snohomish/2026-04-13/drainage.geojson")
obs_gdf      = gpd.read_file("/home/cassandra/Snohomish/2026-04-13/obs.geojson")

In [10]:
refine_specs = [
    ("/home/cassandra/Snohomish/2026-04-13/refine_80m.geojson",     1),
    ("/home/cassandra/Snohomish/2026-04-13/refine_40m.geojson",     2),
    ("/home/cassandra/Snohomish/2026-04-13/channel_buffer.geojson", 2),
    ("/home/cassandra/Snohomish/2026-04-13/refine_20m.geojson",     3),
    ("/home/cassandra/Snohomish/2026-04-13/refine_10m.geojson",     4),
]

refine_parts = []
for fpath, level in refine_specs:
    gdf = gpd.read_file(fpath)
    gdf = gdf[["geometry"]].copy()
    gdf["refinement_level"] = level
    refine_parts.append(gdf)

refinement_gdf = gpd.GeoDataFrame(
    pd.concat(refine_parts, ignore_index=True), crs=include_gdf.crs
)

In [11]:
DEM_PATH = "/home/cassandra/Data/Snoh_DEM_composite/Snohomish_MosaicDEM_modded.tif"
elevation_list = [{"elevation": DEM_PATH}]

In [12]:
FORCING_FILES = [
    "/home/cassandra/Snohomish/2026-04-13/bench_copy/sfincs.bnd",
    "/home/cassandra/Snohomish/2026-04-13/bench_copy/sfincs.bzs",
    "/home/cassandra/Snohomish/2026-04-13/bench_copy/sfincs.src",
    "/home/cassandra/Snohomish/2026-04-13/bench_copy/sfincs.dis",
    "/home/cassandra/Snohomish/2026-04-13/bench_copy/sfincs.obs",
]

In [13]:
MANNING_RANGE = np.linspace(0.01, 0.03, 9)

In [19]:
def gen_test(MANNING_USE):
    
    man_str = f"{MANNING_USE:.4f}"
    out_dir = '/home/cassandra/Snohomish/2026-04-16/' + man_str + '/'

    if not os.path.exists(out_dir):
        os.mkdir(out_dir)
    
    sf = SfincsModel(root=out_dir, mode="w+")
    sf.config.update(
        {
            # --- time ---
            "tref":               "20151225 000000",
            "tstart":             "20161110 000000",
            "tstop":              "20161125 000000",
            "tspinup":            3600,
            # --- output intervals ---
            "dtmapout":           1800,
            "dtmaxout":           86400,
            "dtrstout":           0,
            "trstout":            -999,
            "dthisout":           1800,
            "dtwnd":              1800,
            # --- numerics ---
            "alpha":              0.5,
            "theta":              1,
            "dtmax":              999,
            "huthresh":           0.05,
            "advlim":             9999.9,
            "wiggle_suppression": 1,
            # --- roughness ---
            "manning_land":       MANNING_USE,
            "manning_sea":        MANNING_USE,
            "rgh_lev_land":       0,
            # --- initial conditions ---
            "zsini":              0,
            "qinf":               10,
            # --- physics ---
            "rhoa":               1.25,
            "rhow":               1024,
            "pavbnd":             101200,
            "gapres":             101200,
            "baro":               0,
            "coriolis":           1,
            "latitude":           48,
            "advection":          1,
            "viscosity":          1,
            "nuvisc":             0.01,
            "crsgeo":             0,
            # --- formats ---
            "inputformat":        "bin",
            "outputformat":       "net",
            "bndtype":            1,
            # --- CRS ---
            "epsg":               32610,
            # --- output store flags ---
            "storevelmax":        0,
            "storefluxmax":       1,
            "storevel":           0,
            "storecumprcp":       0,
            "storemeteo":         1,
            "storemaxwind":       0,
            "storetzmax":         0,
            # --- wind drag ---
            "cdnrb":              3,
            "cdwnd":              [0.0, 28.0, 50.0],
            "cdval":              [0.001, 0.0025, 0.0025],
            # --- wave / snapwave ---
            "igperiod":           120,
            "maxlev":             999,
            "stopdepth":          1000,
            "nonh":               0,
            "snapwave_gammaig":   1,
            "snapwave_gammax":    1,
            # --- forcing file references (files copied in step 12) ---
            "bndfile":            "sfincs.bnd",
            "bzsfile":            "sfincs.bzs",
            "srcfile":            "sfincs.src",
            "disfile":            "sfincs.dis",
        }
    )
    
    sf.quadtree_grid.create_from_region(
        region={"geom": include_gdf},
        res=160,
        crs=32610,
        rotated=False,
        refinement_polygons=refinement_gdf,
    )
    
    sf.quadtree_elevation.create(
        elevation_list=elevation_list,
        interp_method="linear",
        buffer_cells=1,
    )
    
    sf.quadtree_mask.create_active(
        include_polygon=include_gdf,
    )
    
    sf.quadtree_mask.create_boundary(
        btype="waterlevel",
        include_polygon=offshore_gdf,
        reset_bounds=True,
    )
    
    sf.quadtree_roughness.create(
        roughness_list=[],
        manning_land=MANNING_USE,
        manning_sea=MANNING_USE,
        rgh_lev_land=0,   # z = 0 m as land/sea boundary
    )
    
    sf.weirs.create(
        locations=levees_gdf,
        dep=DEM_PATH,   # sample crest elevation from DEM
        merge=False,
    )
    
    sf.drainage_structures.create(
        locations=drainage_gdf,
        merge=False,
    )
    
    sf.observation_points.create(
        locations=obs_gdf,
        merge=False,
    )
    
    sf.quadtree_subgrid.create(
        elevation_list=elevation_list,
        roughness_list=[],
        manning_land=MANNING_USE,
        manning_water=MANNING_USE,
        manning_level=0.0,
        nr_subgrid_pixels=40,
        nr_levels=20,
        write_dep_tif=True,    # writes dep_subgrid.tif for inspection
        write_man_tif=False,
        nrmax=5000,
    )
    
    sf.write()
    
    for fname in FORCING_FILES:
        dst = out_dir + fname.split('/')[-1]
        shutil.copy2(fname, dst)

In [20]:
%%capture cap

for MANNING_USE in MANNING_RANGE:
    gen_test(MANNING_USE)

In [36]:
out_dirs  = ['/home/cassandra/Snohomish/2026-04-16/0.0100',
             '/home/cassandra/Snohomish/2026-04-16/0.0125',
             '/home/cassandra/Snohomish/2026-04-16/0.0150',
             '/home/cassandra/Snohomish/2026-04-16/0.0175',
             '/home/cassandra/Snohomish/2026-04-16/0.0200',
             '/home/cassandra/Snohomish/2026-04-16/0.0225',
             '/home/cassandra/Snohomish/2026-04-16/0.0250',
             '/home/cassandra/Snohomish/2026-04-16/0.0275',
             '/home/cassandra/Snohomish/2026-04-16/0.0300']

f_content = """#!/bin/bash

#SBATCH -J {PATH}
#SBATCH -A coastallab
#SBATCH -p cpu-g2
#SBATCH -c 48
#SBATCH -t 04:00:00
#SBATCH -v
#SBATCH -o /gscratch/coastallab/cshender/{PATH}/out.txt

cd /gscratch/coastallab/cshender/{PATH}/
module load singularity
singularity run -B$(pwd) /gscratch/coastallab/cshender/sfincs-cpu.img
"""

run_file_content = ""

for od in out_dirs:
    PATH = od.split('Snohomish/')[-1]
    fc = f_content.format(PATH=PATH)
    sh_file_name = od.split('/')[-1]
    with open('/home/cassandra/Snohomish/'+PATH+'.sh', 'w') as f:
        f.write(fc)
    run_file_content = run_file_content + '\n' + 'sbatch /gscratch/coastallab/cshender/' + PATH + '.sh'

with open('/home/cassandra/Snohomish/2026-04-16/batch_all.sh', 'w') as f:
    f.write(run_file_content)